# Notebook 1 — Prepare final WVS respondent profiles

This notebook prepares the World Values Survey Wave 7 v6.0 respondent profiles used in the LLM experiment.

**Design safeguards**
- Uses the original `Q` variables from the WVS file. It does **not** use the inverted `QP` variables.
- Excludes **Q46 (happiness)** and **Q49 (life satisfaction)** from every LLM profile.
- Requires valid **Q49 = 1–10** and **Q288 = 1–10** for the preregistered analysis sample.
- Converts WVS country codes to country names.
- Includes literacy (`E1_LITERACY`) and settlement type (`H_SETTLEMENT`), matching the earlier wellbeing-LLM profile design.
- Runs automated leakage, coding, country-name, sample-size, and profile checks before saving.

`PILOT_MODE = True` creates the deterministic 1,000-person pilot profile file.  
`PILOT_MODE = False` creates the full preregistered sample (expected N = 93,901).

Do not redistribute the underlying WVS microdata.


In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np

INPUT_FILE = Path("Data/WVS_Cross-National_Wave_7_inverted_csv_v6_0.csv")
OUTPUT_DIR = Path("output/full_wvs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PILOT_MODE = False
PILOT_N = 1000
PILOT_RANDOM_STATE = 20260921

EXPECTED_FULL_N = 93901
PROFILE_VERSION = "wvs7_v6_profiles_v2_2026-09-22"

ALL_OUTPUT = OUTPUT_DIR / "profiles_full_wvs_all.csv"
PILOT_OUTPUT = OUTPUT_DIR / "profiles_for_prediction_PILOT.csv"
FULL_OUTPUT = OUTPUT_DIR / "profiles_for_prediction.csv"
MANIFEST_OUTPUT = OUTPUT_DIR / (
    "profiles_manifest_PILOT.json" if PILOT_MODE else "profiles_manifest_FULL.json"
)

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


In [ ]:
QUESTION_MAPPING = {
    "Q1": "Indicate how important is 'Family' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)",
    "Q2": "Indicate how important is 'Friends' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)",
    "Q3": "Indicate how important is 'Leisure time' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)",
    "Q4": "Indicate how important is 'Politics' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)",
    "Q5": "Indicate how important is 'Work' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)",
    "Q6": "Indicate how important is 'Religion' in your life (1-Very important, 2-Rather important, 3-Not very important, 4-Not at all important)",
    "Q47": "All in all, how would you describe your state of health these days? (1-Very good, 2-Good, 3-Fair, 4-Poor, 5-Very poor)",
    "Q48": "How much freedom of choice and control do you feel you have over the way your life turns out? (1-No choice at all, 10-A great deal of choice)",
    "Q51": "In the last 12 months, how often have you or your family gone without enough food to eat? (1-Often, 2-Sometimes, 3-Rarely, 4-Never)",
    "Q52": "In the last 12 months, how often have you or your family felt unsafe from crime in your home? (1-Often, 2-Sometimes, 3-Rarely, 4-Never)",
    "Q53": "In the last 12 months, how often have you or your family gone without medicine or medical treatment that you needed? (1-Often, 2-Sometimes, 3-Rarely, 4-Never)",
    "Q54": "In the last 12 months, how often have you or your family gone without a cash income? (1-Often, 2-Sometimes, 3-Rarely, 4-Never)",
    "Q55": "In the last 12 months, how often have you or your family gone without a safe shelter over your head? (1-Often, 2-Sometimes, 3-Rarely, 4-Never)",
    "Q56": "Comparing your standard of living with your parents' standard of living when they were about your age (1-Better off, 2-Worse off, 3-About the same)",
    "Q57": "Generally speaking, would you say that most people can be trusted or that you need to be very careful in dealing with people? (1-Most people can be trusted, 2-Need to be very careful)",
    "Q66": "How much confidence do you have in the press? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)",
    "Q69": "How much confidence do you have in the police? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)",
    "Q71": "How much confidence do you have in the government? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)",
    "Q72": "How much confidence do you have in political parties? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)",
    "Q75": "How much confidence do you have in universities? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)",
    "Q76": "How much confidence do you have in elections? (1-A great deal, 2-Quite a lot, 3-Not very much, 4-None at all)",
    "Q106": "Views on income equality (1-Incomes should be made more equal, 10-There should be greater incentives for individual effort)",
    "Q112": "Views on corruption in your country (1-There is no corruption in my country, 10-There is abundant corruption in my country)",
    "Q131": "How secure do you feel these days in your neighborhood? (1-Very secure, 2-Quite secure, 3-Not very secure, 4-Not at all secure)",
    "Q164": "How important is God in your life? (1-Not at all important, 10-Very important)",
    "Q171": "How often do you attend religious services? (1-More than once a week, 2-Once a week, 3-Once a month, 4-Only on special holy days, 5-Once a year, 6-Less often, 7-Never/practically never)",
    "Q173": "Would you say you are (1-A religious person, 2-Not a religious person, 3-An atheist)",
    "Q250": "How important is it for you to live in a country that is governed democratically? (1-Not at all important, 10-Absolutely important)",
    "Q254": "How proud are you to be [country's nationality]? (1-Very proud, 2-Quite proud, 3-Not very proud, 4-Not at all proud, 5-I am not)",
    "Q260": "Respondent's sex (1-Male, 2-Female)",
    "Q262": "Age",
    "Q263": "Were you born in this country or are you an immigrant to this country? (1-Born in this country, 2-Immigrant to this country)",
    "Q269": "Are you a citizen of this country? (1-Yes, 2-No)",
    "Q270": "How many people, including children, live here regularly as members of this household?",
    "Q273": "Current marital status (1-Married, 2-Living together as married, 3-Divorced, 4-Separated, 5-Widowed, 6-Single)",
    "Q274": "Number of children",
    "Q275": "Highest educational level attained (0-Early childhood/no education to 8-Doctoral or equivalent)",
    "Q279": "Employment status (1-Full time, 2-Part time, 3-Self employed, 4-Retired/pensioned, 5-Housewife, 6-Student, 7-Unemployed, 8-Other)",
    "Q281": "Occupational group (0-Never had a job to 10-Farm proprietor/manager)",
    "Q284": "Sector of employment (1-Government/public institution, 2-Private business/industry, 3-Private non-profit organization)",
    "Q285": "Chief wage earner in household (1-Yes, 2-No)",
    "Q287": "Subjective social class (1-Upper class, 2-Upper middle class, 3-Lower middle class, 4-Working class, 5-Lower class)",
    "Q288": "Household income group in your country, counting wages, salaries, pensions and other household income (1-Lowest income group, 10-Highest income group)",
    "Q289": "Religion/denomination (0-None, 1-Roman Catholic, 2-Protestant, 3-Orthodox, 4-Jewish, 5-Muslim, 6-Hindu, 7-Buddhist, 8-Other Christian, 9-Other)",
    "E1": "Respondent's literacy (1-Literate, 2-Illiterate)",
    "H": "Settlement type where the interview was conducted (1-Capital city, 2-Regional center, 3-District center, 4-Another city/town, 5-Village)",
}

# Static mapping so the notebook does not depend on an external country package.
COUNTRY_ALPHA_TO_NAME = {'ABW': 'Aruba', 'AFG': 'Afghanistan', 'AGO': 'Angola', 'AIA': 'Anguilla', 'ALA': 'Åland Islands', 'ALB': 'Albania', 'AND': 'Andorra', 'ARE': 'United Arab Emirates', 'ARG': 'Argentina', 'ARM': 'Armenia', 'ASM': 'American Samoa', 'ATA': 'Antarctica', 'ATF': 'French Southern Territories', 'ATG': 'Antigua and Barbuda', 'AUS': 'Australia', 'AUT': 'Austria', 'AZE': 'Azerbaijan', 'BDI': 'Burundi', 'BEL': 'Belgium', 'BEN': 'Benin', 'BES': 'Bonaire, Sint Eustatius and Saba', 'BFA': 'Burkina Faso', 'BGD': 'Bangladesh', 'BGR': 'Bulgaria', 'BHR': 'Bahrain', 'BHS': 'Bahamas', 'BIH': 'Bosnia and Herzegovina', 'BLM': 'Saint Barthélemy', 'BLR': 'Belarus', 'BLZ': 'Belize', 'BMU': 'Bermuda', 'BOL': 'Bolivia', 'BRA': 'Brazil', 'BRB': 'Barbados', 'BRN': 'Brunei Darussalam', 'BTN': 'Bhutan', 'BVT': 'Bouvet Island', 'BWA': 'Botswana', 'CAF': 'Central African Republic', 'CAN': 'Canada', 'CCK': 'Cocos (Keeling) Islands', 'CHE': 'Switzerland', 'CHL': 'Chile', 'CHN': 'China', 'CIV': "Côte d'Ivoire", 'CMR': 'Cameroon', 'COD': 'Democratic Republic of the Congo', 'COG': 'Congo', 'COK': 'Cook Islands', 'COL': 'Colombia', 'COM': 'Comoros', 'CPV': 'Cabo Verde', 'CRI': 'Costa Rica', 'CUB': 'Cuba', 'CUW': 'Curaçao', 'CXR': 'Christmas Island', 'CYM': 'Cayman Islands', 'CYP': 'Cyprus', 'CZE': 'Czech Republic', 'DEU': 'Germany', 'DJI': 'Djibouti', 'DMA': 'Dominica', 'DNK': 'Denmark', 'DOM': 'Dominican Republic', 'DZA': 'Algeria', 'ECU': 'Ecuador', 'EGY': 'Egypt', 'ERI': 'Eritrea', 'ESH': 'Western Sahara', 'ESP': 'Spain', 'EST': 'Estonia', 'ETH': 'Ethiopia', 'FIN': 'Finland', 'FJI': 'Fiji', 'FLK': 'Falkland Islands (Malvinas)', 'FRA': 'France', 'FRO': 'Faroe Islands', 'FSM': 'Micronesia, Federated States of', 'GAB': 'Gabon', 'GBR': 'Great Britain', 'GEO': 'Georgia', 'GGY': 'Guernsey', 'GHA': 'Ghana', 'GIB': 'Gibraltar', 'GIN': 'Guinea', 'GLP': 'Guadeloupe', 'GMB': 'Gambia', 'GNB': 'Guinea-Bissau', 'GNQ': 'Equatorial Guinea', 'GRC': 'Greece', 'GRD': 'Grenada', 'GRL': 'Greenland', 'GTM': 'Guatemala', 'GUF': 'French Guiana', 'GUM': 'Guam', 'GUY': 'Guyana', 'HKG': 'Hong Kong', 'HMD': 'Heard Island and McDonald Islands', 'HND': 'Honduras', 'HRV': 'Croatia', 'HTI': 'Haiti', 'HUN': 'Hungary', 'IDN': 'Indonesia', 'IMN': 'Isle of Man', 'IND': 'India', 'IOT': 'British Indian Ocean Territory', 'IRL': 'Ireland', 'IRN': 'Iran', 'IRQ': 'Iraq', 'ISL': 'Iceland', 'ISR': 'Israel', 'ITA': 'Italy', 'JAM': 'Jamaica', 'JEY': 'Jersey', 'JOR': 'Jordan', 'JPN': 'Japan', 'KAZ': 'Kazakhstan', 'KEN': 'Kenya', 'KGZ': 'Kyrgyzstan', 'KHM': 'Cambodia', 'KIR': 'Kiribati', 'KNA': 'Saint Kitts and Nevis', 'KOR': 'South Korea', 'KWT': 'Kuwait', 'LAO': 'Laos', 'LBN': 'Lebanon', 'LBR': 'Liberia', 'LBY': 'Libya', 'LCA': 'Saint Lucia', 'LIE': 'Liechtenstein', 'LKA': 'Sri Lanka', 'LSO': 'Lesotho', 'LTU': 'Lithuania', 'LUX': 'Luxembourg', 'LVA': 'Latvia', 'MAC': 'Macao', 'MAF': 'Saint Martin (French part)', 'MAR': 'Morocco', 'MCO': 'Monaco', 'MDA': 'Moldova', 'MDG': 'Madagascar', 'MDV': 'Maldives', 'MEX': 'Mexico', 'MHL': 'Marshall Islands', 'MKD': 'North Macedonia', 'MLI': 'Mali', 'MLT': 'Malta', 'MMR': 'Myanmar', 'MNE': 'Montenegro', 'MNG': 'Mongolia', 'MNP': 'Northern Mariana Islands', 'MOZ': 'Mozambique', 'MRT': 'Mauritania', 'MSR': 'Montserrat', 'MTQ': 'Martinique', 'MUS': 'Mauritius', 'MWI': 'Malawi', 'MYS': 'Malaysia', 'MYT': 'Mayotte', 'NAM': 'Namibia', 'NCL': 'New Caledonia', 'NER': 'Niger', 'NFK': 'Norfolk Island', 'NGA': 'Nigeria', 'NIC': 'Nicaragua', 'NIR': 'Northern Ireland', 'NIU': 'Niue', 'NLD': 'Netherlands', 'NOR': 'Norway', 'NPL': 'Nepal', 'NRU': 'Nauru', 'NZL': 'New Zealand', 'OMN': 'Oman', 'PAK': 'Pakistan', 'PAN': 'Panama', 'PCN': 'Pitcairn', 'PER': 'Peru', 'PHL': 'Philippines', 'PLW': 'Palau', 'PNG': 'Papua New Guinea', 'POL': 'Poland', 'PRI': 'Puerto Rico', 'PRK': 'North Korea', 'PRT': 'Portugal', 'PRY': 'Paraguay', 'PSE': 'Palestine', 'PYF': 'French Polynesia', 'QAT': 'Qatar', 'REU': 'Réunion', 'ROU': 'Romania', 'RUS': 'Russia', 'RWA': 'Rwanda', 'SAU': 'Saudi Arabia', 'SDN': 'Sudan', 'SEN': 'Senegal', 'SGP': 'Singapore', 'SGS': 'South Georgia and the South Sandwich Islands', 'SHN': 'Saint Helena, Ascension and Tristan da Cunha', 'SJM': 'Svalbard and Jan Mayen', 'SLB': 'Solomon Islands', 'SLE': 'Sierra Leone', 'SLV': 'El Salvador', 'SMR': 'San Marino', 'SOM': 'Somalia', 'SPM': 'Saint Pierre and Miquelon', 'SRB': 'Serbia', 'SSD': 'South Sudan', 'STP': 'Sao Tome and Principe', 'SUR': 'Suriname', 'SVK': 'Slovakia', 'SVN': 'Slovenia', 'SWE': 'Sweden', 'SWZ': 'Eswatini', 'SXM': 'Sint Maarten (Dutch part)', 'SYC': 'Seychelles', 'SYR': 'Syria', 'TCA': 'Turks and Caicos Islands', 'TCD': 'Chad', 'TGO': 'Togo', 'THA': 'Thailand', 'TJK': 'Tajikistan', 'TKL': 'Tokelau', 'TKM': 'Turkmenistan', 'TLS': 'Timor-Leste', 'TON': 'Tonga', 'TTO': 'Trinidad and Tobago', 'TUN': 'Tunisia', 'TUR': 'Turkey', 'TUV': 'Tuvalu', 'TWN': 'Taiwan', 'TZA': 'Tanzania', 'UGA': 'Uganda', 'UKR': 'Ukraine', 'UMI': 'United States Minor Outlying Islands', 'URY': 'Uruguay', 'USA': 'United States', 'UZB': 'Uzbekistan', 'VAT': 'Holy See (Vatican City State)', 'VCT': 'Saint Vincent and the Grenadines', 'VEN': 'Venezuela', 'VGB': 'Virgin Islands, British', 'VIR': 'Virgin Islands, U.S.', 'VNM': 'Viet Nam', 'VUT': 'Vanuatu', 'WLF': 'Wallis and Futuna', 'WSM': 'Samoa', 'YEM': 'Yemen', 'ZAF': 'South Africa', 'ZMB': 'Zambia', 'ZWE': 'Zimbabwe'}


In [ ]:
df_raw = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"Raw WVS rows: {len(df_raw):,}")
print(f"Raw WVS columns: {df_raw.shape[1]:,}")

required = [
    "B_COUNTRY_ALPHA", "D_INTERVIEW", "Q49", "Q288",
    "Q260", "Q262", "Q279"
]
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

def clean_numeric(s):
    x = pd.to_numeric(s, errors="coerce")
    return x.mask(x < 0)

df = pd.DataFrame(index=df_raw.index)
df["WVS_ROW_ID"] = np.arange(len(df_raw), dtype=int)

# Retain useful audit metadata when present.
for c in [
    "version", "doi", "A_WAVE", "A_YEAR", "B_COUNTRY", "B_COUNTRY_ALPHA",
    "D_INTERVIEW", "Q_MODE", "W_WEIGHT", "PWGHT", "H_SETTLEMENT", "E1_LITERACY"
]:
    if c in df_raw.columns:
        df[c] = df_raw[c]

# IMPORTANT: use original WVS Q variables only. Do not reconstruct them from QP variables.
survey_qs = [q for q in QUESTION_MAPPING if q.startswith("Q")]
for q in sorted(set(survey_qs) | {"Q46", "Q49"}):
    df[q] = clean_numeric(df_raw[q]) if q in df_raw.columns else np.nan

# Map non-Q profile variables.
df["E1"] = clean_numeric(df_raw["E1_LITERACY"]) if "E1_LITERACY" in df_raw.columns else np.nan
df["H"] = clean_numeric(df_raw["H_SETTLEMENT"]) if "H_SETTLEMENT" in df_raw.columns else np.nan

# Country names for the LLM profile.
alpha = df["B_COUNTRY_ALPHA"].astype("string").str.strip().str.upper()
df["COUNTRY_NAME"] = alpha.map(COUNTRY_ALPHA_TO_NAME)

# If the WVS file already contains a readable country-name field, use it only as a fallback.
if "COUNTRY_NAME" in df_raw.columns:
    fallback = df_raw["COUNTRY_NAME"].astype("string").str.strip()
    df["COUNTRY_NAME"] = df["COUNTRY_NAME"].fillna(fallback)

for w in ["W_WEIGHT", "PWGHT"]:
    if w in df.columns:
        df[w] = pd.to_numeric(df[w], errors="coerce")
        df.loc[df[w] <= 0, w] = np.nan

print("Valid Q49:", f"{df['Q49'].between(1,10).sum():,}")
print("Valid Q288:", f"{df['Q288'].between(1,10).sum():,}")


In [ ]:
ANSWER_LABELS = {
    **{q:{1:"Very important",2:"Rather important",3:"Not very important",4:"Not at all important"} for q in ["Q1","Q2","Q3","Q4","Q5","Q6"]},
    "Q47": {1:"Very good",2:"Good",3:"Fair",4:"Poor",5:"Very poor"},
    **{q:{1:"Often",2:"Sometimes",3:"Rarely",4:"Never"} for q in ["Q51","Q52","Q53","Q54","Q55"]},
    "Q56": {1:"Better off",2:"Worse off",3:"About the same"},
    "Q57": {1:"Most people can be trusted",2:"Need to be very careful"},
    **{q:{1:"A great deal",2:"Quite a lot",3:"Not very much",4:"None at all"} for q in ["Q66","Q69","Q71","Q72","Q75","Q76"]},
    "Q131": {1:"Very secure",2:"Quite secure",3:"Not very secure",4:"Not at all secure"},
    "Q171": {1:"More than once a week",2:"Once a week",3:"Once a month",4:"Only on special holy days",5:"Once a year",6:"Less often",7:"Never, practically never"},
    "Q173": {1:"A religious person",2:"Not a religious person",3:"An atheist"},
    "Q254": {1:"Very proud",2:"Quite proud",3:"Not very proud",4:"Not at all proud",5:"I am not"},
    "Q260": {1:"Male",2:"Female"},
    "Q263": {1:"Born in this country",2:"Immigrant to this country"},
    "Q269": {1:"Yes",2:"No"},
    "Q273": {1:"Married",2:"Living together as married",3:"Divorced",4:"Separated",5:"Widowed",6:"Single"},
    "Q275": {0:"Early childhood education/no education",1:"Primary education",2:"Lower secondary education",3:"Upper secondary education",4:"Post-secondary non-tertiary education",5:"Short-cycle tertiary education",6:"Bachelor or equivalent",7:"Master or equivalent",8:"Doctoral or equivalent"},
    "Q279": {1:"Full time employee",2:"Part time employee",3:"Self employed",4:"Retired/pensioned",5:"Housewife not otherwise employed",6:"Student",7:"Unemployed",8:"Other"},
    "Q281": {0:"Never had a job",1:"Professional and technical",2:"Higher administrative",3:"Clerical",4:"Sales",5:"Service",6:"Skilled worker",7:"Semi-skilled worker",8:"Unskilled worker",9:"Farm worker",10:"Farm proprietor/manager"},
    "Q284": {1:"Government or public institution",2:"Private business or industry",3:"Private non-profit organization"},
    "Q285": {1:"Yes",2:"No"},
    "Q287": {1:"Upper class",2:"Upper middle class",3:"Lower middle class",4:"Working class",5:"Lower class"},
    "Q289": {0:"No denomination",1:"Roman Catholic",2:"Protestant",3:"Orthodox",4:"Jewish",5:"Muslim",6:"Hindu",7:"Buddhist",8:"Other Christian",9:"Other"},
    "E1": {1:"Literate",2:"Illiterate"},
    "H": {1:"Capital city (national capital)",2:"Regional center",3:"District center",4:"Another city/town",5:"Village"},
}

def fmt(v):
    if pd.isna(v):
        return None
    x = float(v)
    return str(int(x)) if x.is_integer() else str(x)

def label(q, v):
    if pd.isna(v):
        return None
    x = int(v) if float(v).is_integer() else v
    return ANSWER_LABELS.get(q, {}).get(x, fmt(v))

def make_profile(row):
    age = fmt(row.get("Q262")) or "unknown age"
    sex = label("Q260", row.get("Q260")) or "person"
    country = row.get("COUNTRY_NAME")
    if pd.isna(country) or not str(country).strip():
        country = "unknown country"

    parts = [f"A {age}-year-old {sex} from {country}"]

    for q, wording in QUESTION_MAPPING.items():
        if q in {"Q260", "Q262"}:
            continue
        v = row.get(q)
        if pd.isna(v):
            continue
        parts.append(
            f"the person answered the question '{wording}' "
            f"as '{fmt(v)} - {label(q, v)}'"
        )

    return parts[0] + (
        " where " + ", ".join(parts[1:]) if len(parts) > 1 else ""
    ) + "."

df["user_description"] = df.apply(make_profile, axis=1)
df["N_PROFILE_ITEMS"] = df[list(QUESTION_MAPPING)].notna().sum(axis=1)
df["PROFILE_VERSION"] = PROFILE_VERSION

eligible = df.loc[
    df["Q49"].between(1, 10) &
    df["Q288"].between(1, 10)
].copy()

print(f"Eligible full sample: {len(eligible):,}")


In [ ]:
# -------------------------
# Automated profile QA
# -------------------------

# The analysis sample must match the preregistered WVS eligibility rule.
if len(eligible) != EXPECTED_FULL_N:
    raise ValueError(
        f"Eligible sample N={len(eligible):,}; expected {EXPECTED_FULL_N:,}. "
        "Stop and investigate before any API calls."
    )

assert eligible["WVS_ROW_ID"].is_unique
assert eligible["Q49"].between(1, 10).all()
assert eligible["Q288"].between(1, 10).all()
assert eligible["user_description"].notna().all()
assert eligible["user_description"].str.len().gt(0).all()

# Every eligible profile must contain its Q288 income response.
income_wording = QUESTION_MAPPING["Q288"]
if not eligible["user_description"].str.contains(income_wording, regex=False).all():
    raise AssertionError("At least one eligible profile is missing Q288.")

# No outcome leakage: Q46/Q49 and their target concepts must not appear in profile text.
leak_patterns = [
    r"\bQ46\b",
    r"\bQ49\b",
    r"life[- ]satisfaction",
    r"satisfied are you with your life",
    r"satisfaction with your life as a whole",
]
for pat in leak_patterns:
    hit = eligible["user_description"].str.contains(pat, case=False, regex=True, na=False)
    if hit.any():
        raise AssertionError(
            f"Outcome leakage pattern {pat!r} found in {int(hit.sum())} profiles."
        )

# All eligible respondents should have a readable country name, not an unmapped alpha code.
bad_country = eligible["COUNTRY_NAME"].isna() | (
    eligible["COUNTRY_NAME"].astype(str).str.upper()
    == eligible["B_COUNTRY_ALPHA"].astype(str).str.upper()
)
if bad_country.any():
    bad_codes = sorted(
        eligible.loc[bad_country, "B_COUNTRY_ALPHA"].dropna().astype(str).unique()
    )
    raise AssertionError(f"Unmapped country codes in eligible sample: {bad_codes}")

# Confirm the corrected items are present in the profile template.
assert "governed democratically" in QUESTION_MAPPING["Q250"]
assert "in your neighborhood" in QUESTION_MAPPING["Q131"]
assert ANSWER_LABELS["Q289"][8] == "Other Christian"
assert ANSWER_LABELS["Q289"][9] == "Other"

print("Automated QA passed.")
print("Eligible countries:", eligible["B_COUNTRY_ALPHA"].nunique())
print("Profile items: median =", int(eligible["N_PROFILE_ITEMS"].median()),
      "| min =", int(eligible["N_PROFILE_ITEMS"].min()),
      "| max =", int(eligible["N_PROFILE_ITEMS"].max()))

display_cols = [
    "WVS_ROW_ID", "B_COUNTRY_ALPHA", "COUNTRY_NAME",
    "Q49", "Q288", "N_PROFILE_ITEMS", "user_description"
]
display(eligible[display_cols].head(3))


In [ ]:
# Save a cleaned audit file locally. Do not redistribute WVS microdata.
df.to_csv(ALL_OUTPUT, index=False)

if PILOT_MODE:
    if len(eligible) < PILOT_N:
        raise ValueError("PILOT_N exceeds eligible sample size.")
    pred = (
        eligible.sample(n=PILOT_N, random_state=PILOT_RANDOM_STATE)
        .sort_values("WVS_ROW_ID")
        .copy()
    )
    out_path = PILOT_OUTPUT
    print(f"PILOT MODE: saving {len(pred):,} respondents.")
else:
    pred = eligible.sort_values("WVS_ROW_ID").copy()
    out_path = FULL_OUTPUT
    print(f"FULL MODE: saving preregistered sample of {len(pred):,} respondents.")

pred.to_csv(out_path, index=False)

manifest = {
    "profile_version": PROFILE_VERSION,
    "pilot_mode": PILOT_MODE,
    "input_file": str(INPUT_FILE),
    "output_file": str(out_path),
    "n_rows": int(len(pred)),
    "n_countries": int(pred["B_COUNTRY_ALPHA"].nunique()),
    "eligibility": "Q49 in 1..10 and Q288 in 1..10",
    "pilot_random_state": PILOT_RANDOM_STATE if PILOT_MODE else None,
    "sha256": sha256_file(out_path),
}
with open(MANIFEST_OUTPUT, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved:", out_path)
print("Manifest:", MANIFEST_OUTPUT)
print("SHA256:", manifest["sha256"])


## Before running Notebook 2

For the pilot:
1. keep `PILOT_MODE = True`;
2. inspect several `user_description` rows;
3. confirm the automated QA cell passes;
4. use the resulting `profiles_for_prediction_PILOT.csv`.

For full data collection:
1. change `PILOT_MODE = False`;
2. rerun Notebook 1 from the top;
3. confirm **N = 93,901** and all QA checks pass;
4. retain the printed SHA256 and manifest with the study materials;
5. then run Notebook 2 in full mode.

The pilot sample is for technical validation only; do not use it to test the preregistered substantive hypotheses.


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=7aaa7215-b731-433d-9b62-8be4a70a4410' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>